In [ ]:
# pip install --quiet  gspread oauth2client gspread_dataframe


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# 1. Biblioteca padrão
import datetime
import re

# 2. Bibliotecas de terceiros
import numpy as np
import pandas as pd
# 3. APIs e utils específicas
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from gspread_dataframe import set_with_dataframe

In [2]:
# Função utilitária para padronizar coluna "Valor" com vírgula decimal
def normalize_valor_column(df, col_name='Valor'):
    """
    — Substitui strings vazias por NaN
    — Remove separador de milhares (pontos)
    — Troca vírgula por ponto decimal
    — Converte em float (errors='coerce' transforma inválidos em NaN)
    """
    s = df[col_name].astype(str).str.strip()
    s = s.replace('', np.nan)
    s = s.str.replace(r'\.', '', regex=True).str.replace(',', '.', regex=False)
    df[col_name] = pd.to_numeric(s, errors='coerce')
    return df

def unquote_iof(obs: str) -> str:
    """
    Se obs tem o padrão IOF de "algo", retorna apenas algo.
    Caso contrário, retorna obs sem alteração.
    """
    # procura IOF de "conteúdo"
    m = re.match(r'IOF de\s*"([^"]+)"', obs)
    if m:
        return m.group(1)
    return obs

def extrair_yelumseg(obs: str) -> str:
    """
    Se 'Yelumseg' estiver na string, retorna apenas 'Yelumseg'.
    Caso contrário, retorna a string original.
    """
    return 'Yelumseg' if 'Yelumseg' in obs else obs

def encontrar_data_recente_com_hora_zero(lista_de_datas: list):
    """
    Identifica o objeto de data mais recente em uma lista que tenha 
    o horário "00:00:00".

    Args:
        lista_de_datas: Uma lista de strings, onde cada string é uma data.
                        Ex: ['26/06/2025 00:00:00', '30/06/2025 21:35:16']

    Returns:
        O objeto datetime mais recente ou uma mensagem de erro em string.
    """
    datas_parseadas = []
    formato = '%d/%m/%Y %H:%M:%S'

    for data_str in lista_de_datas:
        if data_str.strip().endswith('00:00:00'):
            try:
                data_obj = datetime.strptime(data_str, formato)
                datas_parseadas.append(data_obj)
            except ValueError:
                print(f"Aviso: A data '{data_str}' não está no formato esperado e foi ignorada.")

    if datas_parseadas:
        data_mais_recente = max(datas_parseadas)
        # Retorna o objeto de data diretamente, sem formatação
        return data_mais_recente
    else:
        return "Nenhuma data com horário 00:00:00 foi encontrada."


In [3]:
# 1. Autenticação Google Sheets
scope = [
    'https://spreadsheets.google.com/feeds',
    'https://www.googleapis.com/auth/drive'
]
creds = ServiceAccountCredentials.from_json_keyfile_name('../acesso.json', scope)
client = gspread.authorize(creds)

### Codigo para recuperar o backup local, use quando por algum motivo o FinData e o backup remoto estao errados

In [ ]:
# # 3. Leitura do CSV do Nubank (sem parse_dates para não falhar)
# csv_path = '/Users/victorzore/Desktop/aihub_financial/data-transactions/nubank/backup/FinData_backup_20250829_190407.csv'
# bkup = pd.read_csv(
#     csv_path,
#     sep=','     # ajuste se for outro separador
# )

# # Acessa a planilha pelo nome ou ID
# sheet = client.open("FinData").sheet1
# # Apaga os dados existentes antes de salvar
# sheet.clear()

# # Escreve o DataFrame na planilha Google Sheets
# set_with_dataframe(sheet, bkup)

In [ ]:
# # 2. Leitura e pré-processamento do histórico no Google Sheets
# sheet = client.open("Backup").sheet1
# sheet.get_all_records()
# backup = pd.DataFrame(sheet.get_all_records(numericise_ignore=['all']))

# Lendo os dados

In [4]:
sheet = client.open("FinData").sheet1
sheet.get_all_records()
finhistorico = pd.DataFrame(sheet.get_all_records(numericise_ignore=['all']))

csv_path = '/Users/victorzore/Desktop/aihub_financial/data-transactions/nubank/input/nubankdezembro.csv'
df = pd.read_csv(
    csv_path,
    sep=','
)

df_copy = df.copy()
finhistoricocopy = finhistorico.copy()

# Atualizacao do backup

In [5]:
sheet = client.open("Backup").sheet1
sheet.clear()
set_with_dataframe(sheet, finhistorico)

# Tratando os historico

In [6]:
finhistorico = normalize_valor_column(finhistorico, 'Valor')

finhistorico['Observações'] = finhistorico['Observações'].apply(unquote_iof)
finhistorico['Observações'] = finhistorico['Observações'].apply(extrair_yelumseg)
finhistorico['Observações'] = finhistorico['Observações'].str.split(r'\s*-\s*Parcela', regex=True).str[0]

formato_completo = '%d/%m/%Y %H:%M:%S'
formato_apenas_data = '%d/%m/%Y'
finhistorico['Data_tratada'] = pd.to_datetime(finhistorico['Data'], format=formato_completo, errors='coerce')

datas_com_hora_zero = finhistorico[finhistorico['Data_tratada'] == finhistorico['Data_tratada'].dt.normalize()]
ultimadata = max(datas_com_hora_zero['Data_tratada'])

df['title'] = df['title'].str.split(r'\s*-\s*Parcela', regex=True).str[0]
df['title'] = df['title'].apply(unquote_iof)
df['title'] = df['title'].apply(extrair_yelumseg)

df['data_como_timestamp'] = pd.to_datetime(df['date'])
df = df[df['data_como_timestamp'] > ultimadata]

df.columns = ['Data', 'Observações', 'Valor', 'Data_tratada']

historico = finhistorico.copy()
novastransacoes = df.copy()

novastransacoes['Nome'] = 'Victor'


referencia = finhistorico.drop_duplicates(subset=['Nome', 'Observações', 'Grupo'], keep='last')
referencia = referencia[['Nome', 'Observações', 'Grupo']]

# 1) Aplique o groupby e **atribua** o resultado de volta (ou a outra variável)
trans = (
    novastransacoes
      .groupby(
          ['Nome', 'Observações'],
          dropna=False
      )
      .agg(
          Valor=('Valor', 'sum'),                    # soma dos valores
          Data =('Data_tratada', 'max')              # data MAIS RECENTE
      )
      .reset_index()
)

transacoes = trans.merge(referencia, how='left', on=['Nome', 'Observações'], suffixes=('', '_ref'))
transacoes = transacoes[['Data', 'Nome', 'Grupo', 'Valor', 'Observações']]

referencia = finhistorico.drop_duplicates(subset=['Nome', 'Observações', 'Grupo','Tipo'], keep='last')
referencia = referencia[['Nome', 'Observações', 'Grupo','Tipo']]

mask = (
    (referencia['Nome'] == 'Victor') &
    (referencia['Tipo'] == 'Individual') &
    (referencia['Observações'] == 'NuTag*GGJ0C95')
)

# mantém só as linhas que NÃO satisfazem a máscara
referencia = referencia[~mask]

transacoes = transacoes.merge(referencia, how='left', on=['Nome', 'Observações', 'Grupo'])
transacoes = transacoes[['Data', 'Nome', 'Grupo', 'Tipo', 'Valor', 'Observações']]

transacoes = transacoes[transacoes['Valor']>0]

transacoes['Valor'] = transacoes['Valor'].round(0).astype(str).str.replace(r'\.0$', '', regex=True)
transacoes['Data'] = transacoes['Data'].dt.strftime(formato_completo)

# 6. Concatena histórico original + novos registros
final = pd.concat([finhistoricocopy,transacoes], ignore_index=True)

# 7. Gravação de volta no Google Sheets
sheet = client.open("FinData").sheet1
sheet.clear()
set_with_dataframe(sheet, final)

print("Planilha 'FinData' atualizada com sucesso!")

Planilha 'FinData' atualizada com sucesso!


In [7]:
final

,Data,Nome,Grupo,Tipo,Valor,Observações
0,30/06/2025 21:35:16,Victor,Bar/Restaurante,Individual,200,Potencia
1,30/06/2025 21:18:35,Victor,Luz/Internet,Casal,104,
2,30/06/2025 21:17:49,Victor,Presentes,Individual,620,Presente Lala
3,30/06/2025 21:16:58,Victor,Luz/Internet,Casal,155,
4,30/06/2025 21:16:15,Victor,Curso,Individual,339,
...,...,...,...,...,...,...
1460,25/12/2025 00:00:00,Victor,Saúde/Estética,Individual,60,Vila Pompeia Barbearia
1461,27/12/2025 00:00:00,Victor,NaN,NaN,195,Yurichignolli
1462,08/12/2025 00:00:00,Victor,NaN,NaN,54,Zet
1463,29/12/2025 00:00:00,Victor,Delivery,Individual,462,iFood - NuPay
